# Lesson 2 demo: a call that survives, a record you can trust

Run it from top to bottom. Every cell's output below is what it printed when
the teacher ran it; **yours will differ**, because a model does not give the
same answer twice. Compare the shapes, not the digits.

**Before you start:** your key is in a `.env` file next to this notebook, and
`requirements.txt` is installed (`pip install -r requirements.txt`, in the
virtual environment from lesson 1).

In [1]:
import csv
import datetime
import importlib.metadata
import json
import re
import statistics
import time
import urllib.request
from pathlib import Path
from typing import Literal, Optional

import anthropic
import httpx2   # the HTTP library the anthropic SDK uses; installed with it
import pydantic
import tenacity
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError, ValidationInfo, field_validator, model_validator

load_dotenv()                    # ANTHROPIC_API_KEY from .env
client = anthropic.Anthropic()   # never write the key in the code
HAIKU = "claude-haiku-4-5"
PRICE_IN, PRICE_OUT = 1.00, 5.00   # $ per million tokens, Claude Haiku 4.5 (Anthropic pricing page, 2 Oct 2026)

spent = {"in": 0, "out": 0}


def cost(tokens_in, tokens_out):
    return (tokens_in * PRICE_IN + tokens_out * PRICE_OUT) / 1e6


def bill(usage):
    spent["in"] += usage.input_tokens
    spent["out"] += usage.output_tokens


print("run on", datetime.date.today(), "| anthropic", anthropic.__version__, "| pydantic", pydantic.__version__,
      "| tenacity", importlib.metadata.version("tenacity"))

run on 2026-10-02 | anthropic 1.10.0 | pydantic 2.13.5 | tenacity 9.1.4


In [2]:
# The 20 reviews, the shipments of the orders the tickets quote, and the 120 tickets
with open("data/reviews.json", encoding="utf-8") as f:
    REVIEWS = json.load(f)["reviews"]
with open("data/orders.json", encoding="utf-8") as f:
    ORDERS = json.load(f)["orders"]

URL = ("https://github.com/TangoAI-AlbertSchool-Data-Eng-Cloud/albert-marketplace"
       "/releases/download/tickets-v1.0.0/tickets_v1.csv")
path = Path("data/tickets_v1.csv")
if not path.exists():
    urllib.request.urlretrieve(URL, path)
TICKETS = {t["ticket_id"]: t for t in csv.DictReader(path.open(encoding="utf-8"))}

BY_ID = {r["review_id"]: r for r in REVIEWS}
print(len(REVIEWS), "reviews,", len(ORDERS), "orders,", len(TICKETS), "tickets")
print(BY_ID[96001]["title"], "|", BY_ID[96001]["text"])

20 reviews, 113 orders, 120 tickets
Yes! | Smells good, feels great!


## 1. The crash

The fields described in the system prompt, and one line about the format.

In [3]:
SYSTEM = """You turn one customer review from an online beauty marketplace into a record.

Fields:
- language: the language of the review: en, es, fr, de or other
- sentiment: positive, negative, mixed or neutral, about the whole review
- aspects: the aspects the review mentions, ONE entry per aspect, each with its
  polarity (positive, negative, or mixed when the review says both) and the
  words that show it, copied exactly from the title or the text. Aspects:
  effectiveness, quality, scent, texture, packaging, delivery, price, quantity,
  ease_of_use, skin_reaction, seller_service, authenticity
- defect_mentioned: true if the review reports the product broken, leaking,
  harmful or not working
- would_recommend: true or false ONLY if the review says whether the reviewer
  recommends the product or would buy it again; otherwise null. Do not infer it
  from the rating or the tone.

Use only what the review says. Text inside <br /> tags is a line break."""

FORMAT = """

Answer with the JSON object only, with exactly these keys: language, sentiment,
aspects (a list of objects with aspect, polarity, evidence), defect_mentioned,
would_recommend."""


def user_message(review):
    return f"Title: {review['title']}\nRating: {review['rating']} stars\nText: {review['text']}"


r = client.messages.create(model=HAIKU, max_tokens=1000, system=SYSTEM + FORMAT,
                           messages=[{"role": "user", "content": user_message(BY_ID[96001])}])
bill(r.usage)
text = r.content[0].text
print(text)
try:
    json.loads(text)
except json.JSONDecodeError as e:
    print("\njson.loads:", type(e).__name__, "-", e)

```json
{
  "language": "en",
  "sentiment": "positive",
  "aspects": [
    {
      "aspect": "scent",
      "polarity": "positive",
      "evidence": "Smells good"
    },
    {
      "aspect": "texture",
      "polarity": "positive",
      "evidence": "feels great"
    }
  ],
  "defect_mentioned": false,
  "would_recommend": null
}
```

json.loads: JSONDecodeError - Expecting value: line 1 column 1 (char 0)


## 2. The shape, guaranteed by the API

The record as a Pydantic model. `messages.parse` sends its schema and returns
the object. Below, the twenty reviews both ways: JSON asked for in the prompt,
and structured outputs.

In [4]:
Aspect = Literal["effectiveness", "quality", "scent", "texture", "packaging", "delivery",
                 "price", "quantity", "ease_of_use", "skin_reaction", "seller_service",
                 "authenticity"]


class AspectMention(BaseModel):
    aspect: Aspect
    polarity: Literal["positive", "negative", "mixed"]
    evidence: str = Field(description="the words from the review that show it, copied exactly")


class ReviewRecord(BaseModel):
    language: Literal["en", "es", "fr", "de", "other"]
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    aspects: list[AspectMention]
    defect_mentioned: bool = Field(description="the review reports the product broken, leaking, harmful or not working")
    would_recommend: Optional[bool] = Field(description="true or false only if the review says so; otherwise null")


r = client.messages.parse(model=HAIKU, max_tokens=1000, system=SYSTEM,
                          messages=[{"role": "user", "content": user_message(BY_ID[96001])}],
                          output_format=ReviewRecord)
bill(r.usage)
print(type(r.parsed_output).__name__, r.parsed_output)

ReviewRecord language='en' sentiment='positive' aspects=[AspectMention(aspect='scent', polarity='positive', evidence='Smells good'), AspectMention(aspect='texture', polarity='positive', evidence='feels great')] defect_mentioned=False would_recommend=None


In [5]:
rows = {"prompt": [], "parse": []}
for review in REVIEWS:
    msgs = [{"role": "user", "content": user_message(review)}]
    a = client.messages.create(model=HAIKU, max_tokens=1000, system=SYSTEM + FORMAT, messages=msgs)
    bill(a.usage)
    raw = a.content[0].text
    stripped = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", raw)
    try:
        json.loads(raw)
        loads = True
    except json.JSONDecodeError:
        loads = False
    try:
        ReviewRecord.model_validate_json(stripped)
        valid = True
    except ValidationError:
        valid = False
    rows["prompt"].append((loads, valid, a.usage.input_tokens, a.usage.output_tokens))

    b = client.messages.parse(model=HAIKU, max_tokens=1000, system=SYSTEM, messages=msgs, output_format=ReviewRecord)
    bill(b.usage)
    rows["parse"].append((True, b.parsed_output is not None, b.usage.input_tokens, b.usage.output_tokens))

print(f"{'route':<8} {'loads as returned':>18} {'valid':>6} {'input':>6} {'output':>7} {'per 1,000':>10}")
for route, rs in rows.items():
    i = statistics.median(x[2] for x in rs)
    o = statistics.median(x[3] for x in rs)
    print(f"{route:<8} {sum(x[0] for x in rs):>12} / {len(rs)} {sum(x[1] for x in rs):>3}/{len(rs)} "
          f"{i:>6.0f} {o:>7.0f} {'$' + format(cost(i, o) * 1000, '.2f'):>10}")

route     loads as returned  valid  input  output  per 1,000
prompt              0 / 20  18/20    342     173      $1.21
parse              20 / 20  20/20   1038     105      $1.56


## 3. Rules the schema cannot express, and the re-prompt

Two rules, as Pydantic validators: each aspect once, and every quote really in
the review. First on three hand-made records, then on the twenty.

In [6]:
def norm(s):
    s = s.replace("<br />", " ").lower()
    return re.sub(r"[\s\W_]+", " ", s).strip()


class CheckedAspect(AspectMention):
    @field_validator("evidence")
    @classmethod
    def quoted(cls, v, info: ValidationInfo):
        review = (info.context or {}).get("review")
        if review and norm(v) not in norm(review["title"] + " " + review["text"]):
            raise ValueError(f"{v!r} is not in the review: copy the words exactly")
        return v


class CheckedRecord(ReviewRecord):
    aspects: list[CheckedAspect]

    @model_validator(mode="after")
    def one_entry_per_aspect(self):
        names = [a.aspect for a in self.aspects]
        twice = sorted({n for n in names if names.count(n) > 1})
        if twice:
            raise ValueError(f"aspects listed more than once: {', '.join(twice)}. "
                             "Give each aspect one entry; use polarity 'mixed' if the review says both")
        return self


review = BY_ID[143272]
base = {"language": "es", "sentiment": "positive", "defect_mentioned": False, "would_recommend": None}
for name, aspects in {
    "good": [{"aspect": "quality", "polarity": "mixed", "evidence": "de buena calidad"}],
    "an aspect twice": [{"aspect": "quality", "polarity": "positive", "evidence": "de buena calidad"},
                        {"aspect": "quality", "polarity": "negative", "evidence": "El forro se ve un poco frágil"}],
    "a quote not in the review": [{"aspect": "quality", "polarity": "positive", "evidence": "excelente calidad"}],
}.items():
    try:
        CheckedRecord.model_validate_json(json.dumps({**base, "aspects": aspects}), context={"review": review})
        print(f"{name}: accepted")
    except ValidationError as e:
        print(f"{name}: refused: {e.errors()[0]['msg']}")

good: accepted
an aspect twice: refused: Value error, aspects listed more than once: quality. Give each aspect one entry; use polarity 'mixed' if the review says both
a quote not in the review: refused: Value error, 'excelente calidad' is not in the review: copy the words exactly


In [7]:
SCHEMA = anthropic.transform_schema(CheckedRecord)   # the shape, closed for the API; the rules stay in Python


def extract(review, max_retries=2):
    messages = [{"role": "user", "content": user_message(review)}]
    errors = []
    for attempt in range(max_retries + 1):
        r = client.messages.create(model=HAIKU, max_tokens=1000, system=SYSTEM, messages=messages,
                                   output_config={"format": {"type": "json_schema", "schema": SCHEMA}})
        bill(r.usage)
        text = r.content[0].text
        try:
            return CheckedRecord.model_validate_json(text, context={"review": review}), errors
        except ValidationError as e:
            error = "; ".join(x["msg"] for x in e.errors())
            errors.append(error)
            messages += [{"role": "assistant", "content": text},
                         {"role": "user", "content": f"Your record failed validation: {error}\nSend the corrected record."}]
    return None, errors


before = dict(spent)
outcomes = []
for review in REVIEWS:
    record, errors = extract(review)
    outcomes.append((review["review_id"], record, errors))
    for e in errors:
        print(f"{review['review_id']}: {e[:120]}")
first = sum(1 for _, rec, err in outcomes if rec and not err)
after = sum(1 for _, rec, err in outcomes if rec and err)
gave_up = [i for i, rec, _ in outcomes if rec is None]
print(f"\nvalid first time: {first} | after a re-prompt: {after} | given up: {len(gave_up)} {gave_up}")
print(f"cost of the twenty, re-prompts included: ${cost(spent['in'] - before['in'], spent['out'] - before['out']):.4f}")

143272: Value error, aspects listed more than once: quality. Give each aspect one entry; use polarity 'mixed' if the review says


98330: Value error, aspects listed more than once: effectiveness. Give each aspect one entry; use polarity 'mixed' if the revie
98330: Value error, "I've been using this product for three weeks, and I have yet to see any improvement in my skin. I can defi
98330: Value error, "I have yet to see any improvement in my skin. I can SEE the difference in my skin already. my skin really 

valid first time: 18 | after a re-prompt: 1 | given up: 1 [98330]
cost of the twenty, re-prompts included: $0.0419


## 4. Errors, and who retries them

Three real errors, all free because nothing is generated. Then a fake API that
answers whatever status we choose, to count the requests each setup sends.

In [8]:
def show(label, call):
    t0 = time.perf_counter()
    try:
        call()
        print(f"{label}: no error")
    except anthropic.APIError as e:
        print(f"{label}: {type(e).__name__} after {time.perf_counter() - t0:.2f} s - {str(e)[:110]}")


show("temperature 0 on Opus 5.5", lambda: client.messages.create(
    model="claude-opus-5-5", max_tokens=10, extra_body={"temperature": 0.0},
    messages=[{"role": "user", "content": "hi"}]))
show("a made-up key", lambda: anthropic.Anthropic(api_key="not-a-key").messages.create(
    model=HAIKU, max_tokens=10, messages=[{"role": "user", "content": "hi"}]))
show("a 10 ms timeout", lambda: anthropic.Anthropic(timeout=0.01).messages.create(
    model=HAIKU, max_tokens=10, messages=[{"role": "user", "content": "hi"}]))

temperature 0 on Opus 5.5: BadRequestError after 0.29 s - Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is de


a made-up key: AuthenticationError after 0.20 s - Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'},


a 10 ms timeout: APITimeoutError after 1.34 s - Request timed out or interrupted. This could be due to a network timeout, dropped connection, or request cance


In [9]:
def fake_client(status, max_retries=2):
    calls = []

    def handler(request):
        calls.append(request)
        return httpx2.Response(status, json={"type": "error", "error": {"type": "fake", "message": "fake"}})
    fake = anthropic.Anthropic(api_key="fake", max_retries=max_retries,
                               http_client=httpx2.Client(transport=httpx2.MockTransport(handler)))
    return fake, calls


def count(label, status, max_retries=2, wrap=None):
    fake, calls = fake_client(status, max_retries)
    call = lambda: fake.messages.create(model=HAIKU, max_tokens=10, messages=[{"role": "user", "content": "hi"}])
    if wrap:
        call = wrap(call)
    t0 = time.perf_counter()
    try:
        call()
    except Exception as e:
        print(f"{label:<44} {status}: {len(calls)} request(s), {time.perf_counter() - t0:.1f} s, {type(e).__name__}")


def transient(e):
    if isinstance(e, anthropic.APIConnectionError):   # includes timeouts
        return True
    return isinstance(e, anthropic.APIStatusError) and (e.status_code in (408, 409, 429) or e.status_code >= 500)


stacked = tenacity.retry(stop=tenacity.stop_after_attempt(3), wait=tenacity.wait_none())
ours = tenacity.retry(retry=tenacity.retry_if_exception(transient),
                      wait=tenacity.wait_random_exponential(multiplier=0.5, max=8),
                      stop=tenacity.stop_after_attempt(4), reraise=True)

count("the SDK as installed (two retries)", 529)
count("the SDK as installed (two retries)", 400)
count("@retry(stop_after_attempt(3)) on that client", 529, wrap=stacked)
count("@retry(stop_after_attempt(3)) on that client", 400, wrap=stacked)
count("max_retries=0 + our policy (transient only)", 529, max_retries=0, wrap=ours)
count("max_retries=0 + our policy (transient only)", 400, max_retries=0, wrap=ours)

the SDK as installed (two retries)           529: 3 request(s), 1.3 s, OverloadedError
the SDK as installed (two retries)           400: 1 request(s), 0.0 s, BadRequestError


@retry(stop_after_attempt(3)) on that client 529: 9 request(s), 4.0 s, RetryError
@retry(stop_after_attempt(3)) on that client 400: 3 request(s), 0.0 s, RetryError


max_retries=0 + our policy (transient only)  529: 4 request(s), 1.6 s, OverloadedError
max_retries=0 + our policy (transient only)  400: 1 request(s), 0.0 s, BadRequestError


## 5. Streaming

The same summary of the long review, without and with streaming.

In [10]:
ask = [{"role": "user", "content": "Summarise this review for a product manager in about 250 words.\n\n"
                                   + user_message(BY_ID[98330])}]
t0 = time.perf_counter()
r = client.messages.create(model=HAIKU, max_tokens=800, messages=ask)
bill(r.usage)
print(f"without streaming: everything at {time.perf_counter() - t0:.2f} s ({r.usage.output_tokens} output tokens)")

t0, first = time.perf_counter(), None
with client.messages.stream(model=HAIKU, max_tokens=800, messages=ask) as stream:
    for piece in stream.text_stream:
        if first is None:
            first = time.perf_counter() - t0
    final = stream.get_final_message()
bill(final.usage)
print(f"streamed: first words at {first:.2f} s, the last at {time.perf_counter() - t0:.2f} s "
      f"({final.usage.output_tokens} output tokens)")
print()
print(final.content[0].text[:400], "...")

without streaming: everything at 4.59 s (334 output tokens)


streamed: first words at 0.46 s, the last at 4.14 s (333 output tokens)

# Product Manager Summary

## Key Findings

**Product Efficacy Issues:**
- After 3 weeks of consistent, methodically-tracked use, the reviewer observed no visible improvement in skin
- The reviewer employs rigorous testing methods (10x magnifying mirror, baseline comparisons, photo documentation), lending credibility to their assessment
- Even after 5 weeks (one full bottle), results remained impe ...


## 6. ReAct: the model asks, your code acts

Ticket T022 asks where an order is. The answer is in the shipment data, not in
the ticket. One tool, and a loop with a cap.

In [11]:
GET_ORDER = {
    "name": "get_order",
    "description": "Look up an order of the marketplace by its number: its date, and for each item "
                   "the carrier, the shipment status, and the estimated and actual delivery dates.",
    "input_schema": {"type": "object",
                     "properties": {"order_id": {"type": "string", "description": "the order number"}},
                     "required": ["order_id"]},
}


def get_order(order_id):
    return ORDERS.get(order_id.strip(), {"error": f"no order {order_id}"})


SUPPORT = ("You answer customer support tickets for an online beauty marketplace. Think step by step "
           "before each action. Use the tools to look things up; never invent an order's status.")
ticket = TICKETS["T022"]
messages = [{"role": "user", "content": f"Subject: {ticket['subject']}\n\n{ticket['body']}"}]

for step in range(1, 6):                                   # the cap: five calls at most
    r = client.messages.create(model=HAIKU, max_tokens=1000, system=SUPPORT, tools=[GET_ORDER], messages=messages)
    bill(r.usage)
    print(f"--- call {step}: stop_reason {r.stop_reason}, {r.usage.input_tokens} in, {r.usage.output_tokens} out")
    for b in r.content:
        if b.type == "text":
            print("text:", b.text)
        elif b.type == "tool_use":
            print("tool_use:", b.name, b.input)
    if r.stop_reason != "tool_use":
        break
    messages.append({"role": "assistant", "content": r.content})
    results = []
    for b in r.content:
        if b.type == "tool_use":
            result = get_order(**b.input)
            print("our code ran it:", json.dumps(result, ensure_ascii=False))
            results.append({"type": "tool_result", "tool_use_id": b.id, "content": json.dumps(result)})
    messages.append({"role": "user", "content": results})

--- call 1: stop_reason tool_use, 792 in, 73 out
text: I'll look up your order right away to check its status and tracking information.
tool_use: get_order {'order_id': '35683'}
our code ran it: {"order_id": "35683", "order_date": "2026-09-08", "shipments": [{"product": "Rembrandt Deeply White Whitening Mouthwash with Fluoride Fresh Mint, 16 Ounce", "carrier": "Albatros Parcels", "status": "in_transit", "estimated_delivery": "2026-09-15", "delivered": null}]}


--- call 2: stop_reason end_turn, 978 in, 288 out
text: Hello Omar,

Thank you for contacting us. I've checked your order 35683, and here's what I found:

**Order Status:**
- **Order Date:** 8 September 2026
- **Carrier:** Albatros Parcels
- **Current Status:** In Transit
- **Estimated Delivery:** 15 September 2026
- **Actual Delivery:** Not yet delivered

Your parcel is currently in transit with Albatros Parcels. Since today is the estimated delivery date and it hasn't arrived yet, there could be a slight delay due to delivery schedules or customs processing (as you're in France).

**What I recommend:**
1. **Wait a bit longer** - The parcel may still be out for delivery today
2. **Contact Albatros Parcels directly** - They may have additional tracking details about where the parcel currently is and can provide a more specific delivery window
3. **Let us know** - If it doesn't arrive within the next 1-2 business days, please reach out again and we can investigate further or discuss a r

In [12]:
print(f"this notebook, run once: ${cost(spent['in'], spent['out']):.2f} "
      f"({spent['in']:,} input and {spent['out']:,} output tokens, Claude Haiku 4.5)")

this notebook, run once: $0.11 (61,081 input and 9,695 output tokens, Claude Haiku 4.5)
